# Baseline Template

Copy to `notebooks/NN_<your_method>_baseline.ipynb`. The pipeline is visible: load data → select → train → save. Replace the selection call with your own `@register_selection`-decorated function.

Three steps to add a new baseline:

1. Implement a selection function in `src/text_distillation/distillation.py` and decorate it with `@register_selection("your_method")`.
2. Copy this notebook and replace the selection call in step 2.
3. Run `pytest tests/test_selection_registry.py`.

In [ ]:
from pathlib import Path

from text_distillation import (
    TimingTracker,
    load_baseline_data,
    save_baseline_run,
)
from text_distillation.model import (
    load_sequence_classifier,
    load_tokenizer,
    train_text_classifier,
)
from text_distillation.saving import create_run_dir
from text_distillation.distillation import select_stratified_random  # replace with your method

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUNS_DIR = PROJECT_ROOT / "artifacts" / "runs"

In [ ]:
EXPERIMENT_PREFIX = "my_method"
METHOD_NAME = "my_method"
K_PER_CLASS = 100
DATASET_NAMES = ["ag_news"]
MODEL_NAMES = ["bert-base-uncased", "roberta-base"]
SEED = 42

In [ ]:
results = []
for dataset_name in DATASET_NAMES:
    # 1. Data.
    data = load_baseline_data(dataset_name, seed=SEED)
    print(f"[{dataset_name}] train pool: {len(data.train_pool)}, eval: {len(data.eval_dataset)}, labels: {data.label_names}")

    # 2. Selection — replace this call with your own @register_selection function.
    selection_tracker = TimingTracker()
    with selection_tracker.measure("selection_sec"):
        train_dataset = select_stratified_random(
            data.train_pool,
            k_per_class=K_PER_CLASS,
            label_column=data.dataset_info.label_column,
            seed=SEED,
        )
    print(f"  distilled: {len(train_dataset)} examples")

    # 3-5. Train each model on the same distilled subset.
    run_suffix_str = f"k{K_PER_CLASS}"
    for model_name in MODEL_NAMES:
        safe = f"{dataset_name}_{model_name}".replace("/", "_").replace("-", "_")
        run_dir = create_run_dir(RUNS_DIR, f"{EXPERIMENT_PREFIX}_{safe}_{run_suffix_str}")

        tracker = TimingTracker()
        tracker.merge(selection_tracker)

        # 3. Model — load tokenizer + classifier head sized for this dataset.
        tokenizer = load_tokenizer(model_name)
        model = load_sequence_classifier(
            model_name,
            num_labels=data.num_labels,
            label_names=data.label_names,
        )

        # 4. Training.
        _, metrics = train_text_classifier(
            model=model,
            tokenizer=tokenizer,
            train_dataset=train_dataset,
            eval_dataset=data.eval_dataset,
            output_dir=run_dir,
            text_columns=data.dataset_info.text_columns,
            metric_name=data.dataset_info.metric_name,
            seed=SEED,
            tracker=tracker,
        )

        # 5. Save config + metrics + distilled dataset.
        row = save_baseline_run(
            run_dir=run_dir,
            data=data,
            method_name=METHOD_NAME,
            model_name=model_name,
            k_per_class=K_PER_CLASS,
            seed=SEED,
            train_dataset=train_dataset,
            metrics=metrics,
            tracker=tracker,
            project_root=PROJECT_ROOT,
        )
        results.append(row)
        print(f"  [{model_name}] accuracy={metrics['accuracy']:.3f} f1_macro={metrics['f1_macro']:.3f}")
results